In [1]:
import os, json, math, random, gc, time, ast
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0

from tqdm import tqdm

print("✅ Imports successful")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

✅ Imports successful
✅ CUDA available: False


# BirdCLEF 2026 Inference v6 - 2-Model Ensemble (Perch + EfficientNet-B0)
## Strategy:
- Load 5 Perch folds
- Load 5 EfficientNet-B0 folds
- 2-window ensemble (start + middle)
- Average 20 predictions (10 models × 2 windows)
- Apply per-species thresholds

In [2]:
# === SETUP ===
TEST_AUDIO_DIR = "/kaggle/input/competitions/birdclef-2026/test_soundscapes"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CFG = dict(
    sr=16000,
    n_mels=64,
    n_fft=1024,
    hop=320,
    fmin=60,
    seconds=5,
    batch_size=32,
)

print(f"✅ Config set up")
print(f"✅ Device: {device}")

# === CHECK DATA AVAILABILITY ===
import os
if not os.path.exists(TEST_AUDIO_DIR):
    print(f"\n⚠️  WARNING: Test data directory not found!")
    print(f"   Path: {TEST_AUDIO_DIR}")
    print(f"   This notebook is designed to run on Kaggle where test data is available.")
    print(f"   Running locally will produce incomplete/empty predictions.")
    print(f"   Continuing anyway - submission will contain no predictions.\n")
else:
    print(f"✅ Test audio directory found: {TEST_AUDIO_DIR}")

✅ Config set up
✅ Device: cpu
✅ Test audio directory found: /kaggle/input/competitions/birdclef-2026/test_soundscapes


In [3]:
# === LOAD SPECIES & THRESHOLDS ===
# Try to load from Kaggle path first, fall back to local
species_paths = [
    "/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/species.json",
    "species.json"
]

species = None
for path in species_paths:
    try:
        with open(path, "r") as f:
            species = json.load(f)
        print(f"✅ Loaded species from: {path}")
        break
    except FileNotFoundError:
        continue

if species is None:
    print("❌ ERROR: Could not load species.json from any path!")
    raise FileNotFoundError("species.json not found")

# Try to load thresholds from Kaggle path, fall back to defaults
thresholds_path = "/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/optimal_thresholds_v6.json"
try:
    with open(thresholds_path, "r") as f:
        optimal_thresholds = json.load(f)
    print(f"✅ Loaded per-species thresholds")
except FileNotFoundError:
    print(f"⚠️  Using default threshold (0.5) for all species")
    optimal_thresholds = {sp: 0.5 for sp in species}

idx = {lab: i for i, lab in enumerate(species)}
n_classes = len(species)

print(f"✅ Loaded {n_classes} species")
print(f"First 10 species: {species[:10]}")

✅ Loaded species from: /kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/species.json
✅ Loaded per-species thresholds
✅ Loaded 234 species
First 10 species: ['1161364', '116570', '1176823', '1491113', '1595929', '209233', '22930', '22956', '22961', '22967']


In [4]:
# === HELPER FUNCTIONS ===
def fixed_length_mono(y, sr, seconds=5):
    target = sr * seconds
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < target:
        y = np.pad(y, (0, target-len(y)))
    else:
        y = y[:target]
    return y.astype(np.float32)

def logmel_from_wave(wave, sr):
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr, n_fft=CFG["n_fft"], hop_length=CFG["hop"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=sr//2, power=2.0
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min = S_db.min()
    S_max = S_db.max()
    if S_max - S_min < 1e-9:
        S_norm = np.zeros_like(S_db, dtype=np.float32)
    else:
        S_norm = (S_db - S_min) / (S_max - S_min + 1e-9)
    return np.clip(S_norm, 0.0, 1.0).astype(np.float32)

print("✅ Helper functions defined")

✅ Helper functions defined


In [5]:
# === PERCH ARCHITECTURE ===
class PerchAudio(nn.Module):
    """Lightweight CNN optimized for bird audio"""
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        self.head = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.head(x)
        return x

print("✅ Perch model defined")

✅ Perch model defined


In [6]:
# === EFFICIENTNET-B0 ARCHITECTURE ===
class EfficientNetB0Audio(nn.Module):
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        self.model = efficientnet_b0(weights=None)
        self.model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, n_classes)
        )
    
    def forward(self, x):
        features = self.model(x)
        return self.head(features)

print("✅ EfficientNet-B0 model defined")

✅ EfficientNet-B0 model defined


In [7]:
# === LOAD ALL TRAINED MODELS ===
print("Loading all 10 trained models (5 folds × 2 architectures)...")

perch_models = []
efficientnet_models = []

for fold_idx in range(5):
    # Load Perch
    perch_model = PerchAudio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)
    perch_model.load_state_dict(torch.load(f"/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/perch_v6_fold_{fold_idx}.pt", map_location=device))
    perch_model.eval()
    perch_models.append(perch_model)
    
    # Load EfficientNet-B0
    eff_model = EfficientNetB0Audio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)
    eff_model.load_state_dict(torch.load(f"/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/efficientnet_v6_fold_{fold_idx}.pt", map_location=device))
    eff_model.eval()
    efficientnet_models.append(eff_model)

print(f"✅ Loaded {len(perch_models)} Perch folds")
print(f"✅ Loaded {len(efficientnet_models)} EfficientNet-B0 folds")

Loading all 10 trained models (5 folds × 2 architectures)...
✅ Loaded 5 Perch folds
✅ Loaded 5 EfficientNet-B0 folds


In [ ]:
# === TEST AUDIO DATASET (3-WINDOW STRATEGY) ===
class TestAudioDataset(Dataset):
    def __init__(self, audio_path, cfg):
        try:
            y, sr0 = sf.read(audio_path, always_2d=False)
        except:
            print(f"Failed to load: {audio_path}")
            y, sr0 = np.zeros(cfg["sr"] * cfg["seconds"]), cfg["sr"]
        
        if sr0 != cfg["sr"]:
            y = librosa.resample(y, orig_sr=sr0, target_sr=cfg["sr"])
        
        y = fixed_length_mono(y, cfg["sr"], cfg["seconds"])
        self.mel = logmel_from_wave(y, cfg["sr"])
        self.win_frames = int(cfg["seconds"] * cfg["sr"] / cfg["hop"]) + 1
    
    def __len__(self):
        # Three windows per audio (to match Kaggle format: _5, _10, _15)
        return 3
    
    def __getitem__(self, idx):
        T = self.mel.shape[1]
        W = self.win_frames
        
        # Three-window: start, middle, end
        if idx == 0:
            start = 0
        elif idx == 1:
            start = max(0, (T - W) // 2)
        else:  # idx == 2
            start = max(0, T - W)
        
        mel_window = self.mel[:, start:start + W]
        mel_window = np.expand_dims(mel_window, 0)
        return torch.from_numpy(mel_window).float()

print("✅ TestAudioDataset with 3-window strategy defined")

✅ Test dataset class defined


In [ ]:
# === PREDICTION FUNCTION ===
def get_predictions_for_audio(audio_path):
    """Get ensemble predictions from 10 models × 3 windows, return separate predictions per window"""
    
    dataset = TestAudioDataset(audio_path, CFG)
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
    
    window_predictions = []  # Store predictions for each window separately
    
    with torch.no_grad():
        for window_idx, x in enumerate(loader):
            x = x.to(device)
            window_preds = np.zeros((n_classes,), dtype=np.float32)
            
            # Get predictions from all Perch folds
            for perch_model in perch_models:
                logits = perch_model(x)
                probs = torch.sigmoid(logits).cpu().numpy()[0]
                window_preds += probs
            
            # Get predictions from all EfficientNet folds
            for eff_model in efficientnet_models:
                logits = eff_model(x)
                probs = torch.sigmoid(logits).cpu().numpy()[0]
                window_preds += probs
            
            # Average over models only (5 folds × 2 architectures = 10 predictions per window)
            window_preds = window_preds / (5 * 2)
            window_predictions.append(window_preds)
    
    return window_predictions  # Return list of predictions, one per window

print("✅ Prediction function defined")

✅ Prediction function defined


In [ ]:
# === GENERATE PREDICTIONS ===
test_dir = Path(TEST_AUDIO_DIR)

# Search for all common audio formats (not just .ogg)
test_files = []
if test_dir.exists():
    for pattern in ["*.ogg", "*.wav", "*.mp3", "*.flac", "*.m4a"]:
        test_files.extend(sorted(test_dir.glob(pattern)))
    # Remove duplicates and sort
    test_files = sorted(list(set(test_files)))

print(f"Found {len(test_files)} test files in {TEST_AUDIO_DIR}")
if len(test_files) > 0:
    print(f"Audio formats found:")
    from collections import Counter
    formats = Counter([f.suffix for f in test_files])
    for fmt, count in sorted(formats.items()):
        print(f"  {fmt}: {count} files")

if len(test_files) == 0:
    print(f"\n⚠️  WARNING: No test audio files found!")
    print(f"   This notebook requires the Kaggle test_soundscapes directory.")
    print(f"   Running locally will produce an EMPTY submission.")
    print(f"   Submission CSV will have proper structure but NO PREDICTIONS.\n")

# Window offsets for Kaggle format (in seconds)
# Based on sample submission format: row_ids end with _5, _10, _15 (3 windows)
WINDOW_OFFSETS = [5, 10, 15]

predictions = {}

for audio_path in tqdm(test_files, desc="Generating predictions"):
    filename = audio_path.stem
    window_preds_list = get_predictions_for_audio(str(audio_path))
    
    # Create a row for each window with the Kaggle format
    for window_idx, window_preds in enumerate(window_preds_list):
        if window_idx < len(WINDOW_OFFSETS):
            row_id = f"{filename}_{WINDOW_OFFSETS[window_idx]}"
            predictions[row_id] = window_preds
        else:
            # Fallback if more windows than offsets defined
            row_id = f"{filename}_{(window_idx + 1) * 5}"
            predictions[row_id] = window_preds

print(f"\n✅ Generated predictions for {len(predictions)} total rows")
print(f"   ({len(test_files)} files × 3 windows per file)")

Found 0 test files in /kaggle/input/competitions/birdclef-2026/test_soundscapes

⚠️  WARNING: No test audio files found!
   This notebook requires the Kaggle test_soundscapes directory.
   Running locally will produce an EMPTY submission.
   Submission CSV will have proper structure but NO PREDICTIONS.



Generating predictions: 0it [00:00, ?it/s]


✅ Generated predictions for 0 test files


In [ ]:
# === DEBUG: INSPECT PREDICTIONS ===
print("Debugging prediction output...")
print(f"\nTotal predictions generated: {len(predictions)}")

if len(predictions) > 0:
    # Show first 10 row_ids
    first_10_ids = list(predictions.keys())[:10]
    print(f"\nFirst 10 row_ids:")
    for i, rid in enumerate(first_10_ids, 1):
        pred_sample = predictions[rid][:5]  # First 5 species predictions
        print(f"  {i}. {rid} -> species predictions (first 5): {[f'{p:.4f}' for p in pred_sample]}")
    
    # Check if row_ids have expected format
    sample_id = first_10_ids[0]
    has_suffix = sample_id.endswith(('_5', '_10', '_15'))
    print(f"\nSample row_id: {sample_id}")
    print(f"Has window suffix (_5, _10, _15)? {has_suffix}")
    
    # Count rows by window suffix
    from collections import Counter
    suffix_counts = Counter()
    for rid in predictions.keys():
        if rid.endswith('_5'):
            suffix_counts['_5'] += 1
        elif rid.endswith('_10'):
            suffix_counts['_10'] += 1
        elif rid.endswith('_15'):
            suffix_counts['_15'] += 1
        else:
            suffix_counts['other'] += 1
    print(f"\nRow_id breakdown by window suffix:")
    for suffix, count in sorted(suffix_counts.items()):
        print(f"  {suffix}: {count}")
else:
    print("ERROR: No predictions generated!")


In [11]:
# === CHECK TEST METADATA ===
print("Checking test metadata...")
test_meta = None
try:
    test_meta = pd.read_csv("/kaggle/input/competitions/birdclef-2026/test.csv")
    print(f"✅ Test metadata shape: {test_meta.shape}")
    print(f"Test metadata columns: {test_meta.columns.tolist()}")
    print(f"\nFirst 10 rows:")
    print(test_meta.head(10))
except FileNotFoundError:
    print("⚠️  No test.csv found - checking alternative paths...")
    # Check for any CSV in test directory
    import glob
    csv_files = glob.glob("/kaggle/input/competitions/birdclef-2026/*.csv")
    print(f"Available CSV files: {csv_files}")


Checking test metadata...
⚠️  No test.csv found - checking alternative paths...
Available CSV files: ['/kaggle/input/competitions/birdclef-2026/sample_submission.csv', '/kaggle/input/competitions/birdclef-2026/taxonomy.csv', '/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv', '/kaggle/input/competitions/birdclef-2026/train.csv']


In [ ]:
# === COMPARE WITH SAMPLE SUBMISSION ===
print("Comparing generated row_ids with expected Kaggle format...")
print("\nExpected format from sample_submission.csv:")
print("  - Format: <base_id>_<window_offset>")
print("  - Example: BC2026_Test_0001_S05_20250227_010002_5")
print("  - Window offsets: _5, _10, _15 (or multiples of 5)")

# Get all unique base IDs (without window suffix)
import re
base_ids = set()
for rid in predictions.keys():
    # Try to extract base (everything before the last _5, _10, _15)
    match = re.match(r'(.+?)(_\d+)$', rid)
    if match:
        base_ids.add(match.group(1))

print(f"\nUnique base IDs found: {len(base_ids)}")
if len(base_ids) > 0:
    sample_bases = list(base_ids)[:5]
    print(f"Sample base IDs:")
    for base in sample_bases:
        print(f"  - {base}")


In [ ]:
# === APPLY PREDICTIONS & CREATE SUBMISSION ===
import pandas as pd

# Load sample submission to get authoritative column order (234 species)
sample_sub_paths = [
    "/kaggle/input/competitions/birdclef-2026/sample_submission.csv",
]
sample_sub = None
for path in sample_sub_paths:
    try:
        sample_sub = pd.read_csv(path)
        print(f"Loaded sample_submission.csv: {sample_sub.shape}")
        break
    except FileNotFoundError:
        continue

if sample_sub is None:
    raise FileNotFoundError("sample_submission.csv not found!")

# Authoritative column list from Kaggle: ['row_id'] + 234 species in correct order
SUBMIT_COLS = list(sample_sub.columns)
submit_species = SUBMIT_COLS[1:]  # just species names, 234 total
print(f"Expected columns: {len(SUBMIT_COLS)} (1 row_id + {len(submit_species)} species)")

# Build a lookup: species_name -> index in model output
# species list comes from Cell 3 (loaded from species.json - model's training species)
species_to_idx = {sp: i for i, sp in enumerate(species)}
print(f"Model has {len(species)} species, submission needs {len(submit_species)} species")
missing = [s for s in submit_species if s not in species_to_idx]
print(f"Species missing from model (will be set to 0.0): {len(missing)}")

# Build submission rows 
rows = []
for row_id, preds in predictions.items():
    row = {'row_id': row_id}
    for sp in submit_species:
        if sp in species_to_idx:
            row[sp] = float(preds[species_to_idx[sp]])
        else:
            row[sp] = 0.0  # species not in model -> predict absent
    rows.append(row)

# Create DataFrame with correct column order
submission_df = pd.DataFrame(rows, columns=SUBMIT_COLS)

# Ensure correct dtypes
submission_df['row_id'] = submission_df['row_id'].astype(str)
for col in submit_species:
    submission_df[col] = submission_df[col].astype('float64')

# Fill any remaining NaN with 0.0
submission_df = submission_df.fillna(0.0)

print(f"\nsubmission_df created:")
print(f"  Rows: {len(submission_df)}")
print(f"  Cols: {len(submission_df.columns)}")
print(f"  row_id dtype: {submission_df['row_id'].dtype}")
print(f"  species dtype: {submission_df[submit_species[0]].dtype}")
print(f"\nFirst 3 row_ids: {submission_df['row_id'].head(3).tolist()}")

In [ ]:
# === SAVE SUBMISSION ===
import os

submission_path = "/kaggle/working/submission.csv"

# Save
submission_df.to_csv(submission_path, index=False)
print(f"Saved: {submission_path}")

# Verify
file_size_mb = os.path.getsize(submission_path) / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

# Validate
nan_count = submission_df.isna().sum().sum()
min_val = submission_df[submit_species].min().min()
max_val = submission_df[submit_species].max().max()

print(f"Rows: {len(submission_df)}, Cols: {len(submission_df.columns)}")
print(f"NaN values: {nan_count}")
print(f"Value range: [{min_val:.4f}, {max_val:.4f}]")
if nan_count > 0:
    print("WARNING: NaN values in submission!")
if min_val < 0.0 or max_val > 1.0:
    print("WARNING: Values outside [0.0, 1.0] range!")
print("DONE")

In [14]:
# === VALIDATE SUBMISSION FORMAT ===
print("Validating submission format...")

# Check for NaN values
nan_count = submission_df.isna().sum().sum()
if nan_count > 0:
    print(f"⚠️  WARNING: Found {nan_count} NaN values in submission!")
    print(f"NaN per column: {submission_df.isna().sum()}")

# Check data types
wrong_dtypes = []
for col in submission_df.columns:
    expected = 'object' if col == 'row_id' else 'float64'
    if str(submission_df[col].dtype) != expected:
        wrong_dtypes.append((col, submission_df[col].dtype, expected))

if wrong_dtypes:
    print(f"⚠️  WARNING: {len(wrong_dtypes)} columns have wrong dtype!")
    for col, actual, expected in wrong_dtypes[:5]:
        print(f"  {col}: {actual} (expected {expected})")

# Check value ranges for species columns
for col in submission_df.columns[1:]:  # Skip row_id
    min_val = submission_df[col].min()
    max_val = submission_df[col].max()
    if min_val < 0 or max_val > 1:
        print(f"⚠️  WARNING: {col} has values outside [0,1]: min={min_val}, max={max_val}")
        break

print(f"✅ Validation complete")


Validating submission format...
✅ Validation complete
